# LINet 5-Fold Cross-Validation on SUN RGB-D

**Hyperparameter tuning with Ray Tune — each trial runs a full 5-fold CV**

Each trial trains 5 models (one per fold), then reports the **median** accuracy.
Ray Tune runs trials in parallel to explore the search space.

---

## Checklist Before Running:

- [ ] **Enable A100 GPU:** Runtime > Change runtime type > A100
- [ ] **Upload dataset to Drive:** `MyDrive/datasets/sunrgbd_19_traintest.tar.gz`
- [ ] **(Optional)** Upload pretrained weights for transfer learning


## 1. Environment Setup & GPU Verification

In [ ]:
# Check GPU availability and specs
import torch
import subprocess

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

# Check PyTorch and CUDA
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

    # Check if it's A100
    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        print("\n✅ A100 GPU detected - PERFECT for training!")
    elif 'V100' in gpu_name:
        print("\n✅ V100 GPU detected - Good for training (slower than A100)")
    elif 'T4' in gpu_name:
        print("\n⚠️  T4 GPU detected - Will be slower, consider upgrading to A100")
    else:
        print(f"\n⚠️  GPU: {gpu_name} - Consider using A100 for best performance")
else:
    print("\n❌ NO GPU DETECTED!")
    print("Please enable GPU: Runtime → Change runtime type → Hardware accelerator: GPU")
    raise RuntimeError("GPU is required for training")

print("\n" + "=" * 60)

In [ ]:
# Detailed GPU info
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

print("\n✅ Google Drive mounted successfully!")
print(f"\nDrive contents:")
!ls -la /content/drive/MyDrive/ | head -20

## 3. Clone Repository to Local Disk (Fast I/O)

**Important:** We clone to `/content/` (local SSD) instead of Drive for 10-20x faster I/O

**Default:** Clone from GitHub (recommended - always gets latest code)

In [ ]:
import os
from pathlib import Path

# Configuration
PROJECT_NAME = "Multi-Stream-Neural-Networks"
GITHUB_REPO = "https://github.com/clingergab/Multi-Stream-Neural-Networks.git"  # UPDATE THIS
LOCAL_REPO_PATH = f"/content/{PROJECT_NAME}"  # Local copy for fast I/O

print("=" * 60)
print("REPOSITORY SETUP")
print("=" * 60)

# Ensure we're in a valid directory
os.chdir('/content')
print(f"Starting in: {os.getcwd()}")

# Check if repo already exists (same session, rerunning cell)
if Path(LOCAL_REPO_PATH).exists() and Path(f"{LOCAL_REPO_PATH}/.git").exists():
    print(f"\n📁 Repo already exists: {LOCAL_REPO_PATH}")
    print(f"🔄 Pulling latest changes...")

    os.chdir(LOCAL_REPO_PATH)
    !git pull
    print("✅ Repo updated")

# Clone from GitHub (first run)
else:
    # Remove old incomplete copy if exists
    if Path(LOCAL_REPO_PATH).exists():
        print(f"\n🗑️  Removing incomplete repo copy...")
        !rm -rf {LOCAL_REPO_PATH}

    print(f"\n🔄 Cloning from GitHub...")
    print(f"   Repo: {GITHUB_REPO}")
    print(f"   Destination: {LOCAL_REPO_PATH}")

    !git clone {GITHUB_REPO} {LOCAL_REPO_PATH}

    # Verify clone succeeded
    if not Path(LOCAL_REPO_PATH).exists():
        raise RuntimeError(f"Failed to clone repository to {LOCAL_REPO_PATH}")

    print("✅ Repo cloned successfully")
    os.chdir(LOCAL_REPO_PATH)

# Verify repo structure
print(f"\n📂 Repository structure:")
!ls -la {LOCAL_REPO_PATH}

print(f"\n✅ Working directory: {os.getcwd()}")

## 4. Install Dependencies

In [ ]:
# Install required packages
print("Installing dependencies...")

!pip install -q h5py tqdm matplotlib seaborn ray[tune] kornia

# Verify installations
import h5py
import tqdm
import matplotlib
import seaborn
import ray
import kornia

print("✅ All dependencies installed!")
print(f"   h5py: {h5py.__version__}")
print(f"   matplotlib: {matplotlib.__version__}")
print(f"   ray: {ray.__version__}")
print(f"   kornia: {kornia.__version__}")


## 5. Copy SUN RGB-D Dataset to Local Disk

**Performance Note:** Local disk I/O is ~10-20x faster than Drive!

**Dataset:** SUN RGB-D 19-category preprocessed dataset with RGB + Depth


In [ ]:
from pathlib import Path
import os

# Paths
DRIVE_DATASET_TAR = "/content/drive/MyDrive/datasets/sunrgbd_19_traintest.tar.gz"
LOCAL_DATASET_PATH = "/dev/shm/sunrgbd_19_traintest"
LOCAL_TRAINVAL_PATH = "/dev/shm/sunrgbd_19_traintest"

print("=" * 60)
print("SUN RGB-D 19-CATEGORY DATASET SETUP")
print("=" * 60)

if Path(LOCAL_DATASET_PATH).exists():
    print(f"Already on local disk: {LOCAL_DATASET_PATH}")
    train_count = len(list(Path(f"{LOCAL_DATASET_PATH}/train/rgb").glob("*.png")))
    print(f"   Train samples: {train_count}")
elif Path(DRIVE_DATASET_TAR).exists():
    print(f"Found on Drive: {DRIVE_DATASET_TAR}")
    tar_name = Path(DRIVE_DATASET_TAR).name
    local_tar = f"/dev/shm/{tar_name}"
    !rsync -ah --info=progress2 {DRIVE_DATASET_TAR} {local_tar}
    print(f"\nExtracting...")
    !tar -xzf {local_tar} -C /dev/shm/ 2>&1 | grep -v "Ignoring unknown extended header"
    !rm {local_tar}
    train_count = len(list(Path(f"{LOCAL_DATASET_PATH}/train/rgb").glob("*.png")))
    print(f"Extracted. Train samples: {train_count}")
else:
    raise FileNotFoundError(f"Dataset not found at {DRIVE_DATASET_TAR}")

print(f"\nDataset ready at: {LOCAL_DATASET_PATH}")


## 6. Setup Python Path & Import LINet


In [ ]:
import sys
import os

modules_to_reload = [k for k in sys.modules.keys() if k.startswith('src.')]
for module in modules_to_reload:
    del sys.modules[module]

project_root = '/content/Multi-Stream-Neural-Networks'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project structure:")
!ls -la {project_root}/src/models/

print("\nImporting LiNet and dataloaders...")
from src.models.linear_integration.li_net3 import li_resnet18
from src.data_utils.sunrgbd_dataset import get_sunrgbd_dataloaders, SUNRGBDDataset
from src.training.augmentation_config import AugmentationConfig

from ray import train, tune
from ray.tune.schedulers import ASHAScheduler

print("All imports successful!")


## 8b. Hyperparameter Tuning with Ray Tune (5-Fold CV)

- **Parallel Trials:** Each trial runs a full 5-fold CV internally
- **Median Accuracy:** Trial score = median of 5 fold best-accuracies
- **No Scheduler:** Trials run to completion (no early-stopping across trials)
- **Optional Pretrained Weights:** Load Omni backbone for transfer learning


In [ ]:
import os
import time

# 1. Define Paths explicitly
mps_pipe_dir = "/tmp/nvidia-mps"
mps_log_dir = "/tmp/nvidia-log"

# 2. Create the directories (CRITICAL: Daemon fails if log dir doesn't exist)
os.makedirs(mps_pipe_dir, exist_ok=True)
os.makedirs(mps_log_dir, exist_ok=True)

# 3. Set Environment Variables for the current Python process
os.environ["CUDA_MPS_PIPE_DIRECTORY"] = mps_pipe_dir
os.environ["CUDA_MPS_LOG_DIRECTORY"] = mps_log_dir
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

# 4. Configure GPU and Start Daemon using the SAME environment variables
# We use f-strings to pass the python variables into the shell command
print("Setting GPU to Exclusive Process Mode...")
!nvidia-smi -i 0 -c EXCLUSIVE_PROCESS

print("Starting MPS Daemon...")
# We explicitly pass the env vars to the shell command
!export CUDA_MPS_PIPE_DIRECTORY={mps_pipe_dir} && \
 export CUDA_MPS_LOG_DIRECTORY={mps_log_dir} && \
 nvidia-cuda-mps-control -d

# 5. Verify it is running
print("Verifying Daemon Status...")
time.sleep(1) # Give it a second to start
!ps -ef | grep mps

# Check if the pipe file actually exists
if os.path.exists(os.path.join(mps_pipe_dir, "control")):
    print("✅ MPS Control Pipe found. Setup success.")
else:
    print("❌ MPS Control Pipe NOT found. Check /tmp/nvidia-log for errors.")
    # Optional: Print logs if it failed
    !cat {mps_log_dir}/control.log

In [ ]:
import random
import statistics
import numpy as np

import ray
from ray import tune
import torch
from collections import Counter
from sklearn.model_selection import StratifiedKFold

from src.models.linear_integration.li_net3 import li_resnet18
from src.training.optimizers import create_stream_optimizer
from src.training.schedulers import setup_scheduler
from src.data_utils.sunrgbd_dataset import SUNRGBDDataset
from src.training.augmentation_config import AugmentationConfig
from src.utils.seed import set_seed
from src.data_utils.sunrgbd_dataset import _load_norm_stats
from src.models.common.model_helpers import load_pretrained_backbone


def train_linet_5fold(
    config,
    data_root=None,
    norm_stats=None,
    pretrained_weights_path=None,
    seed=42,
):
    """
    Trainable function for Ray Tune — full 5-fold CV per trial.

    Each trial trains 5 fresh models (one per fold), records each fold's
    best val accuracy, and reports the median to Ray Tune.

    Args:
        config: Ray Tune configuration dict with hyperparameters
        data_root: Path to dataset root (with train/ directory)
        norm_stats: Normalization statistics dict
        pretrained_weights_path: Path to pretrained checkpoint (or None)
        seed: Random seed for reproducible folds
    """
    set_seed(seed, deterministic=False)

    # Per-trial augmentation config
    aug_config = AugmentationConfig(
        rgb_aug_prob=config.get("rgb_aug_prob"),
        rgb_aug_mag=config.get("rgb_aug_mag"),
        depth_aug_prob=config.get("depth_aug_prob"),
        depth_aug_mag=config.get("depth_aug_mag"),
    )

    # Load full training set (both instances share the same underlying data)
    train_dataset = SUNRGBDDataset(
        data_root=data_root,
        split='train',
        normalize=False,
        **aug_config.to_dict(),
    )
    val_dataset = SUNRGBDDataset(
        data_root=data_root,
        split='train',
        normalize=False,
    )
    val_dataset.split = 'val'  # Disable augmentation in __getitem__

    all_labels = train_dataset.labels

    # Deterministic 5-fold split (same folds every trial given seed)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    folds = list(skf.split(range(len(all_labels)), all_labels))

    fold_best_accs = []
    fold_best_losses = []
    fold_best_train_accs = []

    for fold_idx, (train_indices, val_indices) in enumerate(folds):
        g = torch.Generator().manual_seed(seed + fold_idx)

        train_subset = torch.utils.data.Subset(train_dataset, train_indices)
        val_subset = torch.utils.data.Subset(val_dataset, val_indices)

        # Stratified sampling
        subset_labels = [all_labels[i] for i in train_indices]
        label_counts = Counter(subset_labels)
        num_samples = len(subset_labels)
        class_weights = {label: num_samples / count for label, count in label_counts.items()}
        sample_weights = torch.tensor(
            [class_weights[label] for label in subset_labels], dtype=torch.float32
        )

        train_sampler = torch.utils.data.WeightedRandomSampler(
            weights=sample_weights,
            num_samples=num_samples,
            replacement=True,
            generator=g,
        )

        def worker_init_fn(worker_id):
            worker_seed = seed + fold_idx * 100 + worker_id
            np.random.seed(worker_seed)
            random.seed(worker_seed)

        train_loader = torch.utils.data.DataLoader(
            train_subset,
            batch_size=config['batch_size'],
            shuffle=False,
            sampler=train_sampler,
            num_workers=1,
            prefetch_factor=2,
            persistent_workers=True,
            pin_memory=True,
            worker_init_fn=worker_init_fn,
        )
        val_loader = torch.utils.data.DataLoader(
            val_subset,
            batch_size=config['batch_size'],
            shuffle=False,
            num_workers=1,
            prefetch_factor=2,
            persistent_workers=False,
            pin_memory=True,
            worker_init_fn=worker_init_fn,
        )

        # Fresh model for each fold
        model = li_resnet18(
            num_classes=19,
            stream_input_channels=[3, 1],
            dropout_p=config["dropout_p"],
            width_multiplier=0.75,
            device="cuda",
            use_amp=True,
        )

        # Load pretrained backbone (same weights for every fold)
        if pretrained_weights_path is not None:
            load_pretrained_backbone(model, pretrained_weights_path, verbose=False)

        optimizer = create_stream_optimizer(
            model,
            optimizer_type='adamw',
            stream_lrs=[config["lr_rgb"], config["lr_depth"]],
            stream_weight_decays=[config["wd_rgb"], config["wd_depth"]],
            shared_lr=config["lr_shared"],
            integration_weight_decay=config["wd_integrated"],
        )

        scheduler = setup_scheduler(
            optimizer,
            scheduler_type='cosine',
            eta_min=[config['s1_eta_min'], config['s2_eta_min'], config['eta_min'], config['eta_min']],
            t_max=config['t_max'],
            train_loader_len=len(train_loader),
            warmup_epochs=5,
            warmup_start_factor=0.2,
        )

        model.compile(
            optimizer=optimizer,
            scheduler=scheduler,
            loss='cross_entropy',
            label_smoothing=config["label_smoothing"],
            gpu_augmentation=True,
            norm_stats=norm_stats,
            **aug_config.to_dict(),
        )

        # Train this fold to completion
        history = model.fit(
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=config['t_max'] + 5,
            early_stopping=True,
            patience=15,
            grad_clip_norm=config["grad_clip_norm"],
            modality_dropout=True,
            modality_dropout_start=config['modality_dropout_start'],
            modality_dropout_ramp=config['modality_dropout_ramp'],
            modality_dropout_rate=config['modality_dropout_rate'],
            verbose=False,
        )

        # Record this fold's best metrics
        fold_best_acc = max(history['val_accuracy'])
        fold_best_loss = min(history['val_loss'])
        fold_best_train = max(history['train_accuracy'])
        fold_best_accs.append(fold_best_acc)
        fold_best_losses.append(fold_best_loss)
        fold_best_train_accs.append(fold_best_train)

        print(f"  Fold {fold_idx}: best_val_acc={fold_best_acc:.4f}, "
              f"epochs={len(history['train_loss'])}")

        # Free GPU memory between folds
        del model, optimizer, scheduler, train_loader, val_loader
        torch.cuda.empty_cache()

    # Compute summary statistics across 5 folds
    median_acc = statistics.median(fold_best_accs)
    mean_acc = statistics.mean(fold_best_accs)
    std_acc = statistics.stdev(fold_best_accs) if len(fold_best_accs) > 1 else 0.0
    median_loss = statistics.median(fold_best_losses)
    median_train_acc = statistics.median(fold_best_train_accs)
    median_gap = median_train_acc - median_acc
    composite = median_acc - 10 * (median_gap ** 3)

    # Report once to Ray Tune (single report per trial)
    tune.report({
        "median_accuracy": median_acc,
        "mean_accuracy": mean_acc,
        "std_accuracy": std_acc,
        "median_loss": median_loss,
        "median_train_acc": median_train_acc,
        "median_gap": median_gap,
        "composite": composite,
        "fold_0_acc": fold_best_accs[0],
        "fold_1_acc": fold_best_accs[1],
        "fold_2_acc": fold_best_accs[2],
        "fold_3_acc": fold_best_accs[3],
        "fold_4_acc": fold_best_accs[4],
    })


In [ ]:
# =============================================================================
# WARM-START CONFIGURATION (Optional)
# =============================================================================
# Restores the HyperOptSearch TPE surrogate model from a previous session
# so exploration continues from where it left off. Trial results are also
# saved to CSV for reference.
# =============================================================================

import hashlib
import json as json_module
import os
import pandas as pd
from pathlib import Path
from ray.tune.search.sample import Domain

WARM_START_ENABLED = True
WARM_START_CSV_PATH = "/content/drive/MyDrive/ray_tune_results/5fold_results.csv"
HYPEROPT_CHECKPOINT_DIR = "/content/drive/MyDrive/ray_tune_results"

# --- Pretrained Weights (Optional) ---
# Set LOAD_WEIGHTS = True to initialize every trial from pretrained backbone
# weights (e.g. from OmniObject3D pretraining). The fc head is skipped
# automatically if num_classes differs.
LOAD_WEIGHTS = False
PRETRAINED_WEIGHTS_PATH = "/content/drive/MyDrive/linet_checkpoints/omni_best/final_model.pt"


def get_search_space_hash(search_space: dict) -> str:
    """Generate a short hash to identify a search space configuration."""

    def _serialize_value(v):
        if isinstance(v, Domain):
            if hasattr(v, "categories"):
                return sorted([repr(c) for c in v.categories])
            sampler_name = type(v.sampler).__name__ if hasattr(v, "sampler") else ""
            domain_str = repr(v.domain_str) if hasattr(v, "domain_str") else type(v).__name__
            return f"{sampler_name}:{domain_str}"
        return repr(v)

    space_repr = {k: _serialize_value(v) for k, v in sorted(search_space.items())}
    space_str = json_module.dumps(space_repr, sort_keys=True)
    return hashlib.md5(space_str.encode()).hexdigest()[:8]


Path(WARM_START_CSV_PATH).parent.mkdir(parents=True, exist_ok=True)

print(f"Warm-start: {'ENABLED' if WARM_START_ENABLED else 'DISABLED'}")
print(f"Load pretrained weights: {'ENABLED' if LOAD_WEIGHTS else 'DISABLED'}")
if LOAD_WEIGHTS:
    print(f"   Weights: {PRETRAINED_WEIGHTS_PATH}")
if WARM_START_ENABLED:
    print(f"   Results CSV: {WARM_START_CSV_PATH}")
    print(f"   HyperOpt checkpoint dir: {HYPEROPT_CHECKPOINT_DIR}")


In [ ]:
# Initialize Ray
from ray.tune.search.hyperopt import HyperOptSearch
from ray.tune.search import ConcurrencyLimiter
from ray.tune import CLIReporter

os.environ["RAY_AIR_NEW_OUTPUT"] = "0"  # must be set BEFORE ray.init()

# os.environ["RAY_TUNE_RESULT_REPORTER_INTERVAL_S"] = "60"

ray.shutdown()  # Clean shutdown of any previous Ray instance
ray.init(
    ignore_reinit_error=True,
    runtime_env={
        "env_vars": {
            "CUDA_MPS_PIPE_DIRECTORY": "/tmp/nvidia-mps",
            "CUDA_MPS_LOG_DIRECTORY": "/tmp/nvidia-log",
            "CUDA_DEVICE_ORDER": "PCI_BUS_ID",
            # Ensure workers see the GPU as Device 0
            "CUDA_VISIBLE_DEVICES": "0"
        }
    }
)

SEED = 42
norm_stats = _load_norm_stats(LOCAL_TRAINVAL_PATH)


# Continuous search space for TPE optimization (80/20 split CV)
search_space = {
    # Learning rates (log-uniform for order-of-magnitude exploration)
    "lr_rgb": tune.uniform(6.5e-5, 7.5e-5),
    "lr_depth": tune.uniform(1.0e-4, 2.0e-4),
    "lr_shared": tune.uniform(1.2e-4, 2.0e-4),

    # Weight decay (log-uniform)
    "wd_rgb": tune.uniform(6.0e-5, 7.0e-5),
    "wd_depth": tune.uniform(4.0e-5, 6.5e-5),
    "wd_integrated": tune.uniform(1.2e-4, 1.5e-4),

    # Scheduler eta_min (log-uniform)
    "s1_eta_min": tune.uniform(9.6e-7, 1.6e-6),
    "s2_eta_min": tune.uniform(2.0e-6, 2.5e-6),
    "eta_min": tune.uniform(6.0e-7, 6.6e-7),

    # Scheduler t_max (integer)
    "t_max": tune.choice([115, 120, 125]),
    "batch_size": tune.choice([64]),

    # Regularization (uniform)
    "dropout_p": tune.uniform(0.515, 0.535),
    "label_smoothing": tune.uniform(0.10, 0.12),
    "grad_clip_norm": tune.uniform(0.75, 0.8),

    # Augmentation parameters (per-stream control)
    "rgb_aug_prob": tune.uniform(0.96, 1.1),
    "rgb_aug_mag": tune.uniform(1.0, 1.1),
    "depth_aug_prob": tune.uniform(1.0, 1.35),
    "depth_aug_mag": tune.uniform(1.13, 1.15),

    # Modality dropout
    "modality_dropout_rate": tune.uniform(0.125, 0.145),
    "modality_dropout_start": tune.choice([0]),
    "modality_dropout_ramp": tune.choice([20]),
}

print("=" * 60)
print("5-FOLD CV SEARCH SPACE (CONTINUOUS, 2-STREAM: RGB + DEPTH)")
print("=" * 60)

# HyperOptSearch uses best_accuracy for the TPE surrogate model
# (so it optimizes for running-best, not noisy per-epoch accuracy)
hyperopt_searcher = HyperOptSearch(
    metric="composite",
    mode="max",
    random_state_seed=SEED,
)

if WARM_START_ENABLED:
    current_hash = get_search_space_hash(search_space)
    hyperopt_checkpoint_path = os.path.join(HYPEROPT_CHECKPOINT_DIR, f"hyperopt_searcher_{current_hash}.pkl")
    _restored = False

    # Migrate old-format checkpoint (hyperopt_searcher.pkl + .hash) to new format
    old_checkpoint = os.path.join(HYPEROPT_CHECKPOINT_DIR, "hyperopt_searcher.pkl")
    old_hash_file = old_checkpoint + ".hash"
    if not os.path.exists(hyperopt_checkpoint_path) and os.path.exists(old_checkpoint) and os.path.exists(old_hash_file):
        with open(old_hash_file, "r") as f:
            old_hash = f.read().strip()
        if old_hash == current_hash:
            os.rename(old_checkpoint, hyperopt_checkpoint_path)
            os.remove(old_hash_file)
            print(f"   Migrated old checkpoint to {hyperopt_checkpoint_path}")

    if os.path.exists(hyperopt_checkpoint_path):
        try:
            hyperopt_searcher.restore(hyperopt_checkpoint_path)
            # Clear stale trial mappings from previous session.
            hyperopt_searcher._live_trial_mapping = {}
            # Remove any instance-level _setup_hyperopt that may have been
            # pickled from a previous session's monkey-patch.
            if '_setup_hyperopt' in hyperopt_searcher.__dict__:
                del hyperopt_searcher.__dict__['_setup_hyperopt']
            # Wrap _setup_hyperopt to preserve restored trials.
            # set_search_properties() (called by Tuner.fit()) triggers
            # _setup_hyperopt() which creates a NEW _hpopt_trials, destroying
            # our restored history. The wrapper re-injects the saved trials.
            _restored_trials = hyperopt_searcher._hpopt_trials
            _original_setup = hyperopt_searcher._setup_hyperopt
            def _patched_setup():
                _original_setup()
                hyperopt_searcher._hpopt_trials = _restored_trials
            hyperopt_searcher._setup_hyperopt = _patched_setup
            # Clear saved search space so Tuner can re-initialize it
            # from param_space via set_search_properties().
            hyperopt_searcher._space = None
            hyperopt_searcher.domain = None
            hyperopt_searcher._points_to_evaluate = None
            n_prev = len(hyperopt_searcher._hpopt_trials.trials)
            print(f"   Restored HyperOptSearch TPE model with {n_prev} previous trials (hash: {current_hash})")
            _restored = True
        except Exception as e:
            print(f"   Failed to restore HyperOptSearch checkpoint: {e}")
            print("   Starting fresh exploration.")
    if not _restored:
        print(f"Starting HyperOptSearch from scratch (hash: {current_hash})")

print(f"\nUsing trainval dataset : {LOCAL_TRAINVAL_PATH}")

reporter = CLIReporter(
    # Explicit column order - LRs first, then other HPs, then metrics
    parameter_columns=[
        "lr_rgb", "lr_depth", "lr_shared",
        "wd_rgb", "wd_depth", "wd_integrated",
        "s1_eta_min", "s2_eta_min", "eta_min",
        "t_max", "batch_size", "dropout_p",
        "label_smoothing", "grad_clip_norm",
        "rgb_aug_prob", "rgb_aug_mag",
        "depth_aug_prob", "depth_aug_mag",
        "modality_dropout_rate", "modality_dropout_start", "modality_dropout_ramp",
    ],

    metric_columns={
        "median_accuracy": "median_acc",
        "mean_accuracy": "mean_acc",
        "std_accuracy": "std_acc",
        "composite": "composite",
    },
    max_report_frequency=30,   # seconds between updates
    print_intermediate_tables=True,

)

# Concurrency limiter for parallel trials
limited_search_alg = ConcurrencyLimiter(
    hyperopt_searcher,
    max_concurrent=10
)

print("=" * 60)
print("STARTING HYPERPARAMETER TUNING")
print("=" * 60)

# Configure Tuner — uses trainval dataset (no val/ split)
tuner = tune.Tuner(
    tune.with_resources(
        tune.with_parameters(
            train_linet_5fold,
            data_root=LOCAL_TRAINVAL_PATH,
            norm_stats=norm_stats,
            seed=SEED,
            pretrained_weights_path=PRETRAINED_WEIGHTS_PATH if LOAD_WEIGHTS else None,
        ),
        resources={"cpu": 1, "gpu": 0.1}
    ),
    param_space=search_space,
    tune_config=tune.TuneConfig(
        # No scheduler — each trial runs all 5 folds to completion
        # and reports once. No intermediate epoch-level early-stopping.
        search_alg=limited_search_alg,
        num_samples=200,
    ),
    run_config=ray.tune.RunConfig(
        progress_reporter=reporter,
        verbose=1,  # integer verbose (not AirVerbosity) forces legacy output engine
    ),
)

# Run Tuning
results = tuner.fit()

# Remove monkey-patched _setup_hyperopt so save() doesn't pickle the closure
if '_setup_hyperopt' in hyperopt_searcher.__dict__:
    del hyperopt_searcher.__dict__['_setup_hyperopt']

# Get Best Result
best_result = results.get_best_result("median_accuracy", "max")

print("\n" + "=" * 60)
print("TUNING COMPLETE")
print("=" * 60)
print(f"Best Trial Config: {best_result.config}")
print(f"Best Trial Median Accuracy: {best_result.metrics['median_accuracy']:.4f}")
print(f"Best Trial Median Loss: {best_result.metrics['median_loss']:.4f}")

In [ ]:
# =============================================================================
# SAVE RESULTS FOR FUTURE WARM-STARTS
# =============================================================================

import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

results_df = results.get_dataframe()

current_hash = get_search_space_hash(search_space)
results_df['search_space_hash'] = current_hash

timestamped_path = f"/content/drive/MyDrive/ray_tune_results/5fold_trial_{timestamp}.csv"
results_df.to_csv(timestamped_path, index=False)
print(f"Saved results to: {timestamped_path}")
print(f"   Search space hash: {current_hash}")

latest_path = WARM_START_CSV_PATH

if os.path.exists(latest_path):
    previous_df = pd.read_csv(latest_path)
    combined_df = pd.concat([previous_df, results_df], ignore_index=True)
    config_cols = [c for c in combined_df.columns if c.startswith('config/')]
    dedup_cols = config_cols + ['search_space_hash']
    combined_df = combined_df.sort_values('median_accuracy', ascending=False)
    combined_df = combined_df.drop_duplicates(subset=dedup_cols, keep='first')
    combined_df.to_csv(latest_path, index=False)

    hash_counts = combined_df['search_space_hash'].value_counts()
    print(f"Updated {latest_path} with {len(results_df)} new trials")
    print(f"   Total unique configs: {len(combined_df)}")
    for h, count in hash_counts.items():
        marker = " (current)" if h == current_hash else ""
        print(f"      {h}: {count} configs{marker}")
else:
    results_df.to_csv(latest_path, index=False)
    print(f"Created {latest_path}")


# =============================================================================
# SAVE HYPEROPTSEARCH CHECKPOINT
# =============================================================================

if WARM_START_ENABLED:
    hyperopt_searcher.save(hyperopt_checkpoint_path)
    n_trials = len(hyperopt_searcher._hpopt_trials.trials)
    print(f"\nSaved HyperOptSearch checkpoint ({n_trials} trials) to {hyperopt_checkpoint_path}")

print(f"\nTo warm-start next run: set WARM_START_ENABLED = True and re-run")


In [ ]:
# Analyze Top 10 Trials from 5-Fold CV
import pandas as pd

print("=" * 80)
print("TOP 10 TRIALS BY MEDIAN ACCURACY (5-FOLD CV)")
print("=" * 80)

df = results.get_dataframe()

df_sorted = df.sort_values('median_accuracy', ascending=False)

display_cols = [
    'median_accuracy', 'mean_accuracy', 'std_accuracy', 'composite',
    'fold_0_acc', 'fold_1_acc', 'fold_2_acc', 'fold_3_acc', 'fold_4_acc',
    'config/lr_rgb', 'config/lr_depth', 'config/lr_shared',
    'config/wd_rgb', 'config/wd_depth', 'config/wd_integrated',
    'config/s1_eta_min', 'config/s2_eta_min', 'config/eta_min', 'config/t_max',
    'config/dropout_p', 'config/label_smoothing', 'config/grad_clip_norm',
    'config/rgb_aug_prob', 'config/rgb_aug_mag',
    'config/depth_aug_prob', 'config/depth_aug_mag',
    'config/modality_dropout_rate', 'config/modality_dropout_start', 'config/modality_dropout_ramp',
]
# Only keep columns that exist
display_cols = [c for c in display_cols if c in df_sorted.columns]

top_10 = df_sorted[display_cols].head(10)

top_10_fmt = top_10.copy()
for col in ['median_accuracy', 'mean_accuracy', 'fold_0_acc', 'fold_1_acc', 'fold_2_acc', 'fold_3_acc', 'fold_4_acc']:
    if col in top_10_fmt.columns:
        top_10_fmt[col] = top_10_fmt[col].apply(lambda x: f"{x*100:.2f}%")
if 'std_accuracy' in top_10_fmt.columns:
    top_10_fmt['std_accuracy'] = top_10_fmt['std_accuracy'].apply(lambda x: f"{x*100:.2f}pp")

sci_cols = [c for c in display_cols if any(k in c for k in ["lr_", "wd_", "eta_min"])]
for col in sci_cols:
    if col in top_10_fmt.columns:
        top_10_fmt[col] = top_10_fmt[col].apply(lambda x: f"{x:.2e}")

float_cols = [c for c in display_cols if c.startswith("config/") and c not in sci_cols and c != "config/t_max"]
for col in float_cols:
    if col in top_10_fmt.columns:
        top_10_fmt[col] = top_10_fmt[col].apply(lambda x: f"{x:.3f}")

print(top_10_fmt.to_string(index=False))
print("\n" + "=" * 80)


In [ ]:
# =============================================================================
# ANALYZE TOP 10 TRIALS BY COMPOSITE (5-FOLD CV)
# =============================================================================

import pandas as pd

RESULTS_CSV_PATH = WARM_START_CSV_PATH
TARGET_HASH = "PASTE_HASH_HERE"  # <-- paste your search_space_hash

df = pd.read_csv(RESULTS_CSV_PATH)

df["search_space_hash"] = (
    df["search_space_hash"].astype(str).str.replace(r"\.0$", "", regex=True)
)
target = str(TARGET_HASH).strip()
df = df[df["search_space_hash"] == target]
print(f"Trials matching hash {target}: {len(df)}")

df = df.sort_values("composite", ascending=False)
top_10 = df.head(10).copy()

config_cols = [c for c in df.columns if c.startswith("config/")]

print("\n" + "=" * 80)
print(f"TOP 10 TRIALS BY COMPOSITE (hash: {target})")
print("=" * 80)

for rank, (_, row) in enumerate(top_10.iterrows(), 1):
    print(f"\n--- #{rank} | Composite: {row['composite']*100:.2f}% | "
          f"Median Acc: {row['median_accuracy']*100:.2f}% | "
          f"Std: {row['std_accuracy']*100:.2f}pp ---")

print("\n" + "=" * 80)
print("HYPERPARAMETER RANGES ACROSS TOP 10")
print("=" * 80)
print(f"{'Parameter':<35} {'Min':>12} {'Max':>12} {'Median':>12}")
print("-" * 75)

for col in config_cols:
    short_name = col.replace("config/", "")
    col_min = top_10[col].min()
    col_max = top_10[col].max()
    col_med = top_10[col].median()
    if abs(col_med) < 0.001:
        print(f"{short_name:<35} {col_min:>12.2e} {col_max:>12.2e} {col_med:>12.2e}")
    else:
        print(f"{short_name:<35} {col_min:>12.4f} {col_max:>12.4f} {col_med:>12.4f}")

print("\n" + "=" * 80)
print("FULL CONFIG TABLE")
print("=" * 80)

display_df = top_10[["composite", "median_accuracy", "mean_accuracy", "std_accuracy"] + config_cols].copy()
display_df.insert(0, "rank", range(1, len(display_df) + 1))
display_df["median_accuracy"] = display_df["median_accuracy"].apply(lambda x: f"{x*100:.2f}%")

sci_cols = [c for c in config_cols if any(k in c for k in ["lr_", "wd_", "eta_min"])]
float_cols = [c for c in config_cols if c not in sci_cols and c != "config/t_max"]

for col in sci_cols:
    if col in display_df.columns:
        display_df[col] = display_df[col].apply(lambda x: f"{x:.2e}")
for col in float_cols:
    if col in display_df.columns:
        display_df[col] = display_df[col].apply(
            lambda x: f"{int(x)}" if col == "config/t_max" else f"{x:.3f}"
        )

display_df.columns = [c.replace("config/", "") for c in display_df.columns]

for col in ["wd_shared", "optimizer_type", "scheduler_type", "epochs"]:
    if col in display_df.columns:
        del display_df[col]

cols = list(display_df.columns)
if "wd_integrated" in cols and "wd_depth" in cols:
    cols.remove("wd_integrated")
    cols.insert(cols.index("wd_depth") + 1, "wd_integrated")
    display_df = display_df[cols]

print(display_df.to_string(index=False))


In [ ]:
# =============================================================================
# ANALYZE FOLD VARIANCE ACROSS TOP TRIALS
# =============================================================================
# Shows how consistent each top config is across folds.
# Low std = reliable config, high std = sensitive to data split.

import pandas as pd

RESULTS_CSV_PATH = WARM_START_CSV_PATH
TARGET_HASH = "PASTE_HASH_HERE"  # <-- paste your search_space_hash

df = pd.read_csv(RESULTS_CSV_PATH)
df["search_space_hash"] = (
    df["search_space_hash"].astype(str).str.replace(r"\.0$", "", regex=True)
)
target = str(TARGET_HASH).strip()
df = df[df["search_space_hash"] == target]

df = df.sort_values("composite", ascending=False)
top_10 = df.head(10).copy()

print("=" * 80)
print(f"FOLD VARIANCE ANALYSIS — TOP 10 TRIALS (hash: {target})")
print("=" * 80)

fold_cols = [c for c in df.columns if c.startswith("fold_") and c.endswith("_acc")]

for rank, (_, row) in enumerate(top_10.iterrows(), 1):
    fold_accs = [row[c] * 100 for c in fold_cols if c in row.index]
    fold_str = " | ".join(f"F{i}={a:.1f}%" for i, a in enumerate(fold_accs))
    print(f"  #{rank} | Median: {row['median_accuracy']*100:.2f}% | "
          f"Std: {row['std_accuracy']*100:.2f}pp | {fold_str}")

print("\n" + "=" * 80)
